# Mrk 421 Fermi-LAT 月 bin QPO 分析

本 notebook 聚焦 `run.py` 已生成的 Fermi-LAT 月 bin 光变，并沿着 “数据读取 -> CWT -> Emmanoulopoulos 零假设 -> GWS 显著性” 的顺序整理分析流程。

原始探索版本已归档至 `archive/notebooks/mkn421_monthly_raw_20260320.ipynb`。局域 2D 显著性与官方 weekly 光变的交叉测试没有删除，但被保留在归档 notebook 中，以避免主流程继续混入未收敛的实验性单元。

In [ ]:

from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.io import fits
import pycwt as wavelet

def locate_project_root(*markers):
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path('/home/server/projects/QPO'),
        Path('/mnt/mydisk/server/projects/PQO'),
    ]
    for cand in candidates:
        if all((cand / marker).exists() for marker in markers):
            return cand
    raise FileNotFoundError(f'Cannot locate project root for markers: {markers}')

PROJECT_ROOT = locate_project_root('mkn421/mkn421/4fgl_j1104.4+3812_lightcurve.fits')
LIGHTCURVE_PATH = PROJECT_ROOT / 'mkn421' / 'mkn421' / '4fgl_j1104.4+3812_lightcurve.fits'
EMMA_PATH = PROJECT_ROOT / 'mkn421' / 'emmanoulopoulos'
if str(EMMA_PATH) not in sys.path:
    sys.path.insert(0, str(EMMA_PATH))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('LIGHTCURVE_PATH =', LIGHTCURVE_PATH)


## 1. 数据读取与月光变

输入文件是 `run.py` 与 `fermipy.lightcurve()` 生成的月 bin FITS 光变。这里先统一读取、清洗并画出用于周期搜索的时间序列。

In [ ]:

with fits.open(LIGHTCURVE_PATH) as hdul:
    lc = hdul[1].data

t_mjd = 0.5 * (lc['tmin'] + lc['tmax']) / 86400.0 + 51910.0
flux = np.asarray(lc['flux'], dtype=float)
flux_err = np.asarray(lc['flux_err'], dtype=float)

mask = np.isfinite(t_mjd) & np.isfinite(flux) & np.isfinite(flux_err)
t_mjd = t_mjd[mask]
flux = flux[mask]
flux_err = flux_err[mask]

plt.figure(figsize=(12, 4))
plt.errorbar(t_mjd, flux, yerr=flux_err, fmt='o', ms=3, capsize=2, color='0.2')
plt.xlabel('Time (MJD)')
plt.ylabel('Flux (ph cm$^{-2}$ s$^{-1}$, 0.1–300 GeV)')
plt.title('Mrk 421 Monthly Light Curve (Fermi-LAT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. 连续小波变换（CWT）

该部分对等间隔月 bin 光变进行 Morlet CWT，得到时频图与 Global Wavelet Spectrum (GWS)。COI 仍保留在图中，用于后续全局显著性的解释。

In [ ]:

y = flux - np.mean(flux)
y /= np.std(y)

dt = np.median(np.diff(t_mjd))
mother = wavelet.Morlet(6)
dj = 1 / 12
s0 = 2 * dt
J = int(np.log2(len(y) * dt / s0) / dj)

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(
    y, dt, dj=dj, s0=s0, J=J, wavelet=mother
)

power = np.abs(wave) ** 2
period = 1 / freqs
T, P = np.meshgrid(t_mjd, period)
GWS = np.mean(power, axis=1)

fig = plt.subplots(figsize=(14, 5))[0]

ax1 = fig.add_subplot(121)
levels = np.linspace(power.min(), power.max(), 50)
im = ax1.contourf(T, P, power, levels=levels, extend='both', cmap='magma')
ax1.fill_between(t_mjd, period.max(), coi, color='white', alpha=0.6, hatch='/')
ax1.set_yscale('log')
ax1.set_ylim(period.min(), period.max())
ax1.set_xlabel('Time [MJD]')
ax1.set_ylabel('Period (day)')
ax1.set_title('Wavelet Power Spectrum (Morlet)')
fig.colorbar(im, ax=ax1, label='Power')

ax2 = fig.add_subplot(122, sharey=ax1)
ax2.plot(GWS, period, color='black')
ax2.set_xscale('log')
ax2.set_xlabel('Global Wavelet Power')
ax2.set_title('Global Wavelet Spectrum')
plt.setp(ax2.get_yticklabels(), visible=False)

plt.tight_layout()
plt.show()


## 3. 零假设模拟：Emmanoulopoulos sampler

这里将观测月光变封装为 `LC` 对象，并拟合其 PDF/PSD，用于后续红噪声背景下的经验显著性评估。

In [ ]:

from emmanoulopoulos.lightcurve import LC
from emmanoulopoulos.emmanoulopoulos_lc_simulation import Emmanoulopoulos_Sampler

time = (t_mjd - np.min(t_mjd)) * u.day
lc_obs = LC(time, flux, tbin=dt * u.day)
lc_obs.fit_PSD()
lc_obs.fit_PDF()

sampler = Emmanoulopoulos_Sampler()
lc_sim = sampler.sample_from_lc(lc_obs)
print('One simulation length =', len(lc_sim.original_flux))


## 4. 模拟质量控制（PDF / CDF / PSD）

先用少量样本检查模拟光变是否大致复现观测的振幅分布与功率谱形状，再决定是否进入大样本显著性计算。

In [ ]:

plt.figure(figsize=(6, 4))
plt.hist(lc_obs.original_flux, bins=30, alpha=0.5, label='obs')
plt.hist(lc_sim.original_flux, bins=30, alpha=0.5, label='sim')
plt.xlabel('Flux')
plt.ylabel('Counts')
plt.legend()
plt.tight_layout()
plt.show()

x_obs = np.sort(lc_obs.original_flux)
F_obs = np.arange(1, len(x_obs) + 1) / (len(x_obs) + 1)
x_sim = np.sort(lc_sim.original_flux)
F_sim = np.arange(1, len(x_sim) + 1) / (len(x_sim) + 1)

plt.figure(figsize=(6, 4))
plt.plot(x_obs, F_obs, label='obs')
plt.plot(x_sim, F_sim, label='sim')
plt.xlabel('Flux')
plt.ylabel('CDF')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

n_sim = 100
sim_lcs = []
for _ in range(n_sim):
    sim_lcs.append(sampler.sample_from_lc(lc_obs))

flux_obs = lc_obs.original_flux
flux_sims_all = np.concatenate([lc.original_flux for lc in sim_lcs])

bins = np.linspace(
    min(flux_obs.min(), flux_sims_all.min()),
    max(flux_obs.max(), flux_sims_all.max()),
    20,
)

plt.figure(figsize=(6, 4))
plt.hist(flux_obs, bins=bins, density=True, alpha=0.5, label='obs')
plt.hist(flux_sims_all, bins=bins, density=True, alpha=0.5, label='sim (100x)')
plt.xlabel('Flux')
plt.ylabel('PDF')
plt.legend()
plt.tight_layout()
plt.show()

f_obs, P_obs = lc_obs.periodogram(window=True)
f1 = np.asarray(f_obs[1:].value)
P_obs1 = np.asarray(P_obs[1:].value) if hasattr(P_obs[1:], 'unit') else np.asarray(P_obs[1:])

P_sims = []
for lc_item in sim_lcs:
    f_sim, P_sim = lc_item.periodogram(window=True)
    assert np.allclose(f_sim, f_obs)
    P_sims.append(P_sim[1:])

P_sims = np.vstack(P_sims)
P_sim_mean = np.mean(P_sims, axis=0)
P_sim_p16 = np.percentile(P_sims, 16, axis=0)
P_sim_p84 = np.percentile(P_sims, 84, axis=0)

plt.figure(figsize=(6, 4))
plt.loglog(f1, P_obs1, 'k', lw=2, label='obs')
plt.loglog(f1, np.asarray(P_sim_mean), 'C1', lw=2, label='sim mean (100)')
plt.fill_between(f1, np.asarray(P_sim_p16), np.asarray(P_sim_p84), color='C1', alpha=0.3, label='sim 16–84%')
plt.xlabel('Frequency [1/day]')
plt.ylabel('Power')
plt.legend()
plt.tight_layout()
plt.show()


## 5. GWS 显著性流程

下面保留原 notebook 的主显著性思路：先以模拟样本建立每个 period 上的 GWS 分布，再给出 pre-trial 与 post-trial 量化。`Nsim=5000` 的单元较耗时，适合在确认环境与依赖正常后再运行。

In [ ]:
# 编写GWS函数
import numpy as np
import pycwt as wavelet

def compute_gws(flux, dt, dj=1/12, s0=None, J=None, mother=None, standardize=True):
    """
    给定 flux（1D array）返回 (period, GWS)
    period 单位是 day（因为 dt 是 day）
    """
    if mother is None:
        mother = wavelet.Morlet(6)
    if s0 is None:
        s0 = 2 * dt
    if J is None:
        J = int(np.log2(len(flux) * dt / s0) / dj)

    y = np.asarray(flux, float)

    if standardize:
        y = y - np.mean(y)
        y = y / np.std(y)

    w, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(y, dt, dj=dj, s0=s0, J=J, wavelet=mother)
    power = np.abs(w) ** 2
    period = 1.0 / freqs
    gws = np.mean(power, axis=1)
    return period, gws


In [ ]:
#　计算观测的GWS + 10个模拟光变的GWS
dt = np.median(np.diff(t_mjd))  # day
mother = wavelet.Morlet(6)
dj = 1/12
s0 = 2 * dt
J = int(np.log2(len(flux) * dt / s0) / dj)
# 计算观测
period_obs, gws_obs = compute_gws(flux, dt, dj=dj, s0=s0, J=J, mother=mother, standardize=True)

import matplotlib.pyplot as plt

n_show = 20
plt.figure(figsize=(6, 5))

# 观测 GWS
plt.plot(gws_obs, period_obs, 'k', lw=2, label="obs")

# 10 条模拟 GWS
for i in range(n_show):
    flux_sim = sim_lcs[i].original_flux
    period_sim, gws_sim = compute_gws(flux_sim, dt, dj=dj, s0=s0, J=J, mother=mother, standardize=True)

    # 确保 period 网格一致（正常情况下会一致）
    if not np.allclose(period_sim, period_obs):
        raise RuntimeError("period grid mismatch: check dt/dj/s0/J consistency")

    plt.plot(gws_sim, period_sim, alpha=0.4)

plt.xscale("log")
plt.yscale("log")
# plt.gca().invert_yaxis()  # 可选：让短周期在上面（按你习惯）
plt.xlabel("Global Wavelet Power")
plt.ylabel("Period (day)")
plt.title("GWS: obs vs 10 simulations")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 生成5000条模拟，对每条计算GWS，计算pre-trail
from tqdm import tqdm
import numpy as np

n_sim = 5000

GWS_sims = []

for i in tqdm(range(n_sim)):
    lc_sim = sampler.sample_from_lc(lc_obs)
    flux_sim = lc_sim.original_flux

    period_sim, gws_sim = compute_gws(
        flux_sim,
        dt,
        dj=dj,
        s0=s0,
        J=J,
        mother=mother,
        standardize=True
    )

    # 确保 period 网格一致
    if i == 0:
        period_ref = period_sim
    else:
        if not np.allclose(period_sim, period_ref):
            raise RuntimeError("Period grid mismatch")

    GWS_sims.append(gws_sim)

GWS_sims = np.array(GWS_sims)   # shape = (5000, n_period)
print(GWS_sims.shape)


In [ ]:
# 计算mc百分数+画阈值曲线
# pre-trial 阈值
gws_95  = np.percentile(GWS_sims, 85, axis=0)
gws_99  = np.percentile(GWS_sims, 98, axis=0)
gws_997 = np.percentile(GWS_sims, 99.9, axis=0)


plt.figure(figsize=(6, 6))

# 观测
plt.plot(gws_obs, period_obs, 'k', lw=2, label="obs")

# pre-trial 阈值
plt.plot(gws_95,  period_obs, '--', label="85% (pre-trial)")
plt.plot(gws_99,  period_obs, '--', label="98% (pre-trial)")
plt.plot(gws_997, period_obs, '--', label="99.9% (pre-trial)")

plt.xscale("log")
plt.yscale("log")
# plt.gca().invert_yaxis()

plt.xlabel("Global Wavelet Power")
plt.ylabel("Period (day)")
plt.title("Global Wavelet Spectrum with pre-trial significance")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 通过卡方拟合计算分位数
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma, norm

# 目标 sigma（单边）
sigmas = np.array([1, 3, 5], dtype=float)
q = norm.cdf(sigmas)   # 单边阈值对应的CDF分位点

# 结果数组
thr_1s = np.empty(GWS_sims.shape[1])
thr_3s = np.empty(GWS_sims.shape[1])
thr_5s = np.empty(GWS_sims.shape[1])

# 可选：存拟合得到的自由度 k（Ren 文里用的那个）
k_eff = np.empty(GWS_sims.shape[1])

for j in range(GWS_sims.shape[1]):
    x = GWS_sims[:, j]
    x = x[np.isfinite(x)]
    x = x[x > 0]  # Gamma/chi2 要求正数

    # 拟合 gamma：返回 shape(a), loc, scale(theta)
    # 建议把 loc 固定为 0（波功率不应有平移项）
    a_hat, loc_hat, scale_hat = gamma.fit(x, floc=0)

    # 等效自由度（如果你想报告/检查）
    k_eff[j] = 2 * a_hat

    # 用拟合分布求 1/3/5σ 的分位数阈值
    thr_1s[j], thr_3s[j], thr_5s[j] = gamma.ppf(q, a=a_hat, loc=0, scale=scale_hat)

print("median k_eff =", np.median(k_eff))


In [ ]:
plt.figure(figsize=(6, 6))

plt.plot(gws_obs, period_obs, 'k', lw=2, label="obs")

plt.plot(thr_1s, period_obs, '--', label="1σ (pre-trial, χ²-fit)")
plt.plot(thr_3s, period_obs, '--', label="3σ (pre-trial, χ²-fit)")
plt.plot(thr_5s, period_obs, '--', label="5σ (pre-trial, χ²-fit)")

plt.xscale("log")
plt.yscale("log")
# 你想大周期在上可以取消注释
# plt.gca().invert_yaxis()

plt.xlabel("Global Wavelet Power")
plt.ylabel("Period (day)")
plt.title("GWS with pre-trial significance (χ² fit)")
plt.legend()
plt.tight_layout()
plt.show()


### 计算post-trail
**pre-trail** 显著性计算了
> 在某个周期上，通过蒙卡模拟得到的power和观测的power比较

接下来做 **post-trial（全局）显著性**，本质是在回答：

> 我在很多个周期上都看了一遍（相当于做了很多次检验），那么观测里出现一个这么高的峰，在纯噪声下发生概率

根据Helena 2023 采用Auchère et al. (2016)的经验公式
$$
P_G= 1-(1-P_L^a)^n
$$
其中：

* (P_L)：**pre-trial（局部）概率**
* (P_G)：**post-trial（全局）概率**
* (n)：**等效 trial 数**


等效trail数n **没有简单地用 period bin 数**，而是用 Auchère 给的经验标定：

$$
\begin{aligned}
a &= 0.805+0.45 \times 2^{-S_{out}\delta j}
\\
n &= 1.136(-S_{out}\delta j)^{1.2}
\end{aligned}
$$

**变量含义：**

* $S_{ out}$：在 global wavelet spectrum 中，位于 COI 之外的“有效周期（scale）bin 数.
* $\delta j$：wavelet scale resolution（这里是 `dj = 1/12`）


流程图：

1. 在某个 period 上，观测 GWS 得到一个 power
2. 用模拟的 χ² 拟合 → 得到 **pre-trial 概率 (P_L)**
3. 计算 $N_{\rm out}$、$\delta j$
4. 用 Auchère 公式得到 (n)
5. 用
   $$
   P_G = 1 - (1 - P_L^a)^n
   $$
   得到 **post-trial 概率**
6. 把 $P_G$ 转成 σ

In [ ]:
# 先用mc百分位数，或者卡方拟合得到pre-trail p-vlaue
# GWS_sims: (Nsim, Np), gws_obs: (Np,)
PL = (np.sum(GWS_sims >= gws_obs[None, :], axis=0) + 1) / (GWS_sims.shape[0] + 1)
# PL 是每个 period bin 的 pre-trial p-value

# 统计在COI之外的有效time bin数
# period: (Np,), coi: (Nt,)
outside = period[:, None] <= coi[None, :]        # (Np, Nt) True=COI外
frac_out = outside.mean(axis=1)                  # 每个 period 在 COI外的时间占比
valid = frac_out > 0                             # 至少有一部分时间在 COI外
S_out = int(valid.sum())

# Auchère经验公式获得n
dj = 1/12
a = 0.805 + 0.45 * 2**(-S_out * dj)
n = 1.136 * (S_out * dj)**1.2

# 把pre-trail 转换成 post-trial
PG = 1.0 - (1.0 - PL**a)**n   # shape (Np,)

# 把post-trial 的p-value转行成sigma
from scipy.stats import norm
sigma_post = norm.isf(PG)   # isf(p)=Phi^{-1}(1-p)
k_star = np.argmin(PG)
print("Best period:", period[k_star])
print("pre-trial p:", PL[k_star], " post-trial p:", PG[k_star], " post-trial sigma:", sigma_post[k_star])



## 6. 归档说明

- 原 notebook 中关于局域 2D wavelet significance 的若干单元依赖未闭合变量，并与当前主线重复较多，故保留在归档 notebook 中。
- 官方 weekly Fermi 光变的交叉测试已由 `mkn421_lhaaso/mkn421_fermi.ipynb` 承担，这里不再重复展开。